In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(torch.cuda.is_available())  # must be True
print(torch.cuda.get_device_name(0))

In [ ]:
!pip install transformers -q
print("✅ Libraries ready")

In [ ]:
import pandas as pd

BASE = '/content/drive/MyDrive/MultimodalSentiment/data/'

train_df = pd.read_csv(BASE + 'train_sent_emo.csv')
val_df   = pd.read_csv(BASE + 'dev_sent_emo.csv')
test_df  = pd.read_csv(BASE + 'test_sent_emo.csv')

print(f"Train: {len(train_df)}")
print(f"Val:   {len(val_df)}")
print(f"Test:  {len(test_df)}")
print(f"\nSentiment classes: {train_df['Sentiment'].unique()}")
print(f"\nClass distribution:\n{train_df['Sentiment'].value_counts()}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
import torchvision.transforms as transforms

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}

img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class MELDDataset(Dataset):
    def __init__(self, df, max_len=128):
        self.df = df[df['Sentiment'].isin(label_map.keys())].reset_index(drop=True)
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Text tokens
        text = str(row['Utterance'])
        enc = tokenizer(text, max_length=self.max_len,
                        padding='max_length', truncation=True,
                        return_tensors='pt')
        input_ids      = enc['input_ids'].squeeze()
        attention_mask = enc['attention_mask'].squeeze()

        # Blank image tensor (no image files in MELD CSV version)
        image = torch.zeros(3, 224, 224)

        label = torch.tensor(label_map[row['Sentiment']], dtype=torch.long)
        return input_ids, attention_mask, image, label

# Create datasets
train_ds = MELDDataset(train_df)
val_ds   = MELDDataset(val_df)
test_ds  = MELDDataset(test_df)

# Create loaders
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, num_workers=2)

print(f"✅ DataLoaders ready")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

In [ ]:
import torch.nn as nn
from transformers import BertModel

class TextSentimentModel(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids,
                        attention_mask=attention_mask)
        cls = out.pooler_output          # shape: (batch, 768)
        cls = self.dropout(cls)
        return self.classifier(cls)      # shape: (batch, 3)

text_model = TextSentimentModel()
total_params = sum(p.numel() for p in text_model.parameters())
print(f"✅ BERT model built")
print(f"Total parameters: {total_params:,}")

In [ ]:
from sklearn.metrics import accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
text_model = text_model.to(device)

optimizer = torch.optim.Adam(text_model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

train_losses, val_losses = [], []
best_val_acc = 0

print(f"Training on: {device}")
print("="*50)

for epoch in range(5):
    # ── Train ──
    text_model.train()
    total_loss = 0

    for batch_idx, (input_ids, attention_mask, _, labels) in enumerate(train_loader):
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels         = labels.to(device)

        optimizer.zero_grad()
        outputs = text_model(input_ids, attention_mask)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        # Progress update every 50 batches
        if (batch_idx + 1) % 50 == 0:
            print(f"  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # ── Validate ──
    text_model.eval()
    preds, true = [], []
    val_loss = 0

    with torch.no_grad():
        for input_ids, attention_mask, _, labels in val_loader:
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels         = labels.to(device)
            outputs        = text_model(input_ids, attention_mask)
            val_loss      += criterion(outputs, labels).item()
            preds         += outputs.argmax(1).cpu().tolist()
            true          += labels.cpu().tolist()

    val_acc = accuracy_score(true, preds)
    val_losses.append(val_loss / len(val_loader))

    print(f"\nEpoch {epoch+1}/5 | Train Loss: {avg_train_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(text_model.state_dict(),
                   '/content/drive/MyDrive/MultimodalSentiment/models/best_text.pth')
        print(f"  ✅ Best model saved! Acc: {best_val_acc:.4f}")

    print("-"*50)

print(f"\n🎯 Best BERT Val Accuracy: {best_val_acc:.4f}")

In [ ]:
import torchvision.models as models

class ImageSentimentModel(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        resnet = models.resnet50(pretrained=True)

        # Freeze all layers except layer4 and fc
        for name, param in resnet.named_parameters():
            if 'layer4' not in name and 'fc' not in name:
                param.requires_grad = False

        # Replace final FC layer
        resnet.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(2048, num_classes)
        )
        self.model = resnet

    def forward(self, images):
        return self.model(images)

img_model = ImageSentimentModel()
trainable = sum(p.numel() for p in img_model.parameters() if p.requires_grad)
print(f"✅ ResNet-50 model built")
print(f"Trainable parameters: {trainable:,}")

In [ ]:
img_model = img_model.to(device)

optimizer_img = torch.optim.Adam(
    filter(lambda p: p.requires_grad, img_model.parameters()), lr=1e-4)

img_train_losses, img_val_losses = [], []
best_img_acc = 0

print(f"Training ResNet-50 on: {device}")
print("="*50)

for epoch in range(5):
    # ── Train ──
    img_model.train()
    total_loss = 0

    for input_ids, attention_mask, images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer_img.zero_grad()
        outputs = img_model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer_img.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    img_train_losses.append(avg_loss)

    # ── Validate ──
    img_model.eval()
    preds, true = [], []

    with torch.no_grad():
        for _, _, images, labels in val_loader:
            images  = images.to(device)
            labels  = labels.to(device)
            outputs = img_model(images)
            preds  += outputs.argmax(1).cpu().tolist()
            true   += labels.cpu().tolist()

    val_acc = accuracy_score(true, preds)
    img_val_losses.append(val_acc)

    print(f"Epoch {epoch+1}/5 | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_img_acc:
        best_img_acc = val_acc
        torch.save(img_model.state_dict(),
                   '/content/drive/MyDrive/MultimodalSentiment/models/best_image.pth')
        print(f"  ✅ Best model saved! Acc: {best_img_acc:.4f}")

print(f"\n🎯 Best ResNet Val Accuracy: {best_img_acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# BERT loss
axes[0].plot(train_losses, label='Train Loss', color='#378ADD', marker='o')
axes[0].plot(val_losses,   label='Val Loss',   color='#D85A30', marker='o')
axes[0].set_title('BERT — Loss Curves')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

# ResNet accuracy
axes[1].plot(img_val_losses, label='Val Accuracy', color='#1D9E75', marker='o')
axes[1].set_title('ResNet-50 — Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/MultimodalSentiment/outputs/day2_curves.png', dpi=150)
plt.show()
print("✅ Curves saved!")

In [ ]:
print("="*45)
print(f"{'Model':<25} {'Val Accuracy':>12}")
print("="*45)
print(f"{'Text only (BERT)':<25} {best_val_acc:>12.4f}")
print(f"{'Image only (ResNet-50)':<25} {best_img_acc:>12.4f}")
print(f"{'Fusion (Day 3)':<25} {'TBD':>12}")
print("="*45)

In [ ]:
import os, json, shutil
from getpass import getpass

# Clear notebook outputs before pushing
notebook_path = "/content/drive/MyDrive/Colab Notebooks/Day2_Text_Image_Models.ipynb"

with open(notebook_path, 'r') as f:
    nb = json.load(f)

for cell in nb['cells']:
    if 'outputs' in cell:
        cell['outputs'] = []
    if 'execution_count' in cell:
        cell['execution_count'] = None

with open(notebook_path, 'w') as f:
    json.dump(nb, f)

print("✅ Outputs cleared!")

# Copy to repo
REPO = "/content/-multimodal-sentiment-analysis"
if not os.path.exists(REPO):
    !git clone https://github.com/Ravitiw0009/-multimodal-sentiment-analysis.git {REPO}

shutil.copy(notebook_path, f"{REPO}/notebooks/Day2_Text_Image_Models.ipynb")
print("✅ Notebook copied!")

# Push
os.chdir(REPO)
!git config --global user.email "your_github_email@gmail.com"
!git config --global user.name "Ravitiw0009"
!git add notebooks/
!git commit -m "Day 2 - BERT and ResNet baselines complete"

TOKEN = getpass("Enter GitHub token: ")
os.system(f"git remote set-url origin https://{TOKEN}@github.com/Ravitiw0009/-multimodal-sentiment-analysis.git")
os.system("git push origin main")
print("✅ Day 2 pushed to GitHub!")